# W12 - Exercise

use the dataset __healthcare_dataset__ for this exercise

Column details: <br>
- **Name**: This column represents the name of the patient associated with the healthcare record. <br>
- **Age**: The age of the patient at the time of admission, expressed in years. <br>
- **Gender**: Indicates the gender of the patient, either "Male" or "Female."<br>
- **Blood** Type: The patient's blood type, which can be one of the common blood types (e.g., "A+", "O-", etc.).<br>
- **Medical** Condition: This column specifies the primary medical condition or diagnosis associated with the patient, such as "Diabetes," "Hypertension," "Asthma," and more.<br>
- **Date of Admission**: The date on which the patient was admitted to the healthcare facility.<br>
- **Doctor**: The name of the doctor responsible for the patient's care during their admission.<br>
- **Hospital**: Identifies the healthcare facility or hospital where the patient was admitted.<br>
- **Insurance Provider**: This column indicates the patient's insurance provider, which can be one of several options, including "Aetna," "Blue Cross," "Cigna," "UnitedHealthcare," and "Medicare."<br>
- **Billing Amount**: The amount of money billed for the patient's healthcare services during their admission. This is expressed as a floating-point number.<br>
- **Room Number**: The room number where the patient was accommodated during their admission.<br>
- **Admission Type**: Specifies the type of admission, which can be "Emergency," "Elective," or "Urgent," reflecting the circumstances of the admission.<br>
- **Discharge Date**: The date on which the patient was discharged from the healthcare facility, based on the admission date and a random number of days within a realistic range.<br>
- **Medication**: Identifies a medication prescribed or administered to the patient during their admission. Examples include "Aspirin," "Ibuprofen," "Penicillin," "Paracetamol," and "Lipitor."<br>
- **Test Results**: Describes the results of a medical test conducted during the patient's admission. Possible values include "Normal," "Abnormal," or "Inconclusive," indicating the outcome of the test.<br>

This exercise is applicable to multiple machine learning algorithms. Create EDA to visualize data and results and explore as far as you can.
Task:
1. Data Cleaning, Transformation
2. Visualization (EDA)
3. Use at least one machine learning techniques (Regression, Classification, Clustering)
4. Use different modelling algorithm to compare results (XGBoost, KNN, Hierarchical, Apriori, Linear, Logistic, etc)

This exercise is to test your willingness to explore, critical thinking and the knowledge when to use what algorithms to achieve a purpose. 

# Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from kneed import KneeLocator
from sklearn.neighbors import KNeighborsClassifier
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.cluster import AgglomerativeClustering
from xgboost import XGBClassifier
from sklearn.preprocessing import OneHotEncoder

In [ ]:
df = pd.read_csv('healthcare_dataset.csv')
df.head()


# 1. Data Cleaning, Transformation

In [ ]:
df = df.drop(['Name', 'Date of Admission', 'Doctor', 'Hospital', 'Insurance Provider', 'Billing Amount', 'Room Number', 'Admission Type', 'Discharge Date'], axis=1)
df.head()

In [ ]:
df_null = df.isna().sum()
df_null

In [ ]:
le = LabelEncoder()
category = ['Gender', 'Blood Type', 'Medical Condition','Medication', 'Test Results']
new_df = df.copy()

for col in category:
    new_df[col] = le.fit_transform(new_df[col])

new_df.head()


In [ ]:
new_df.describe()

# 2. Visualization (EDA)

In [ ]:
columns = ['Gender', 'Blood Type', 'Medical Condition', 'Medication', 'Test Results']
fig, axis = plt.subplots((len(columns) + 2) // 3, 3, figsize=(25, 15))
for i, col_name in enumerate(columns):
    df[col_name].value_counts().plot(kind='pie', autopct='%1.1f%%', pctdistance=0.85, ax=axis[i // 3, i % 3])
    axis[i // 3, i % 3].set_title(col_name)
    axis[i // 3, i % 3].set_title('')

plt.show()

#  Clustering method

In [ ]:
X = new_df.iloc[:, :-1].values
y = new_df.iloc[:, 1].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 32)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
dfs = scaler.fit_transform(new_df)

In [ ]:
wcss = []
for i in range(1,11) :
  km = KMeans(n_clusters=i, max_iter=500, random_state=32)
  km.fit(dfs)
  wcss.append(km.inertia_)
plt.plot(range(1,11), wcss, marker = '.')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')
plt.show()

In [ ]:
findknee = KneeLocator(range(1,11), wcss, curve="convex", direction="decreasing")
print("the best k for your model is ",findknee.elbow)

In [ ]:
kmeans = KMeans(n_clusters = 4, random_state=1)
y_kmeans = kmeans.fit_predict(dfs)
y_kmeans

In [ ]:
agg_clustering = AgglomerativeClustering(n_clusters=4, metric='euclidean', linkage='ward')
labels = agg_clustering.fit_predict(dfs)
df['Cluster'] = labels
df.head()

In [ ]:
df.groupby('Cluster').agg(count = ('Cluster', 'count'),
                                    Age_mean = ('Age', 'mean'),
                                    Gender = ('Gender', pd.Series.mode),
                                    BloodType = ('Blood Type', pd.Series.mode),
                                    MedicalCondition = ('Medical Condition', pd.Series.mode),
                                    medication = ('Medication', pd.Series.mode),
                                    TestResults = ('Test Results', pd.Series.mode))

In [ ]:
x = new_df.drop('Test Results', axis=1).values
y = new_df['Test Results'].values

xTrain, xTest, yTrain, yTest = train_test_split(x, y, test_size=0.2, random_state=0)

In [ ]:
xgBoost = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42, objective='multi:softmax')
xgBoost.fit(xTrain, yTrain)
yPred = xgBoost.predict(xTest)
print('Accuracy score: ', accuracy_score(yTest, yPred))